# J11 — Original thesis five-seed supervised HAM / BCN predictions

## 1. Cohort

In [1]:
import os,io,json,hashlib,subprocess,zipfile,time,gc
from pathlib import Path
import numpy as np,pandas as pd,tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC'];SEEDS=[21,35,42,52,62]
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
OUT=WORK/'j11-original-supervised-source-predictions'
for sub in ['reports','tables','predictions']:(OUT/sub).mkdir(parents=True,exist_ok=True)
TEMP=Path('/kaggle/temp/j11' if Path('/kaggle/temp').exists() else '/mnt/data/j11_temp');TEMP.mkdir(parents=True,exist_ok=True)
for dev in tf.config.list_physical_devices('GPU'):
    try:tf.config.experimental.set_memory_growth(dev,True)
    except RuntimeError:pass

def req(v,s):
    if not bool(v):raise RuntimeError(s)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def write(p,x):Path(p).write_text(json.dumps(x,indent=2,default=str))
def find_cohort():
    found=list(INPUT.rglob('g02a_verified_test_cohort.csv'))
    req(len(found)==1,'Attach exactly one completed G02-A output including g02a_verified_test_cohort.csv')
    p=found[0];d=pd.read_csv(p,dtype=str,keep_default_na=False)
    req({'hf_id','image_id','label','source','label_id'}.issubset(d.columns),'Wrong G02-A cohort schema')
    req(d.hf_id.is_unique and d.image_id.is_unique,'Duplicated mapped identities')
    req(d.source.isin(['HAM10000','BCN20000']).all(),'Source values invalid')
    return d,p
cohort,cohort_path=find_cohort();cohort['label_id']=cohort.label_id.astype(int)
req((cohort.label==cohort.label_id.map(dict(enumerate(CLASSES)))).all(),'G02-A class order mismatch')
print('Verified cohort:',len(cohort),cohort.source.value_counts().to_dict())


Verified cohort: 2239 {'BCN20000': 1240, 'HAM10000': 999}


## 2. Trained checkpoints and original Hugging Face images

In [2]:
# Locate the exact original 5-seed weights from supervised-classification-for-journal.
ckpts={}
for seed in SEEDS:
    f=[p for p in INPUT.rglob('best_by_val_bacc.weights.h5') if p.parent.name==f'seed_{seed}' and 'mbec_plain_supervised_resnet50' in str(p)]
    req(len(f)==1,f'Seed {seed}: attach original supervised notebook output; discovered {f}')
    ckpts[seed]=f[0]
HF_URL='https://huggingface.co/datasets/MKZuziak/ISIC_2019_224/resolve/main/data.zip'
found=[p for p in INPUT.rglob('data.zip') if 'isic' in str(p).lower() and p.is_file()]
req(len(found)<=1,f'Ambiguous HF archives: {found[:5]}')
zip_path=found[0] if found else TEMP/'data.zip'
if not zip_path.is_file():
    r=subprocess.run(['curl','-fL','--retry','6','--connect-timeout','30','-o',str(zip_path),HF_URL],check=False)
    req(r.returncode==0 and zip_path.stat().st_size>50*10**6,'HF 224 image ZIP download failed; enable Kaggle Internet')
with zipfile.ZipFile(zip_path) as z:
    entries={}
    for m in z.infolist():
        if m.filename.lower().endswith(('.jpg','.jpeg','.png')) and not m.is_dir():
            entries.setdefault(Path(m.filename).stem,[]).append(m.filename)
    member={};ambiguous=[]
    for ident in cohort.hf_id:
        matches=entries.get(ident,[])
        if not matches:ambiguous.append({'hf_id':ident,'reason':'ABSENT'});continue
        if len(matches)==1:member[ident]=matches[0];continue
        dig={}
        for name in matches:dig.setdefault(hashlib.sha256(z.read(name)).hexdigest(),[]).append(name)
        if len(dig)==1:member[ident]=sorted(matches)[0]
        else:ambiguous.append({'hf_id':ident,'reason':'DISTINCT_DUPLICATE_STEMS','members':';'.join(matches)})
    pd.DataFrame(ambiguous,columns=['hf_id','reason','members']).to_csv(OUT/'reports/ambiguous_hf_members.csv',index=False)
req(len(member)>0,'No original HF image identities could be resolved')
cohort=cohort.loc[cohort.hf_id.isin(member)].reset_index(drop=True)
print('Identities retained:',len(cohort),'unresolved:',len(ambiguous))
# Extract only verified evaluation images. Never choose among different-content duplicate names.
image_paths=[]
with zipfile.ZipFile(zip_path) as z:
    for i,row in enumerate(cohort.itertuples()):
        p=TEMP/'verified_images'/f'{row.hf_id}.jpg';p.parent.mkdir(parents=True,exist_ok=True)
        if not p.is_file():p.write_bytes(z.read(member[row.hf_id]))
        image_paths.append(str(p))
cohort['filepath']=image_paths


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   974  100   974    0     0   3202      0 --:--:-- --:--:-- --:--:--  3203
100  136M  100  136M    0     0  64.0M      0  0:00:02  0:00:02 --:--:-- 88.0M


Identities retained: 2239 unresolved: 0


## 3. Frozen five-seed inference and export

In [3]:
def build(seed):
    tf.keras.utils.set_random_seed(seed)
    image=keras.Input((224,224,3),name='image')
    x=layers.Lambda(lambda z:tf.keras.applications.resnet50.preprocess_input(tf.cast(z,tf.float32)*255.),name='resnet50_preprocess')(image)
    resnet=ResNet50(include_top=False,weights=None,input_shape=(224,224,3),pooling='avg')
    x=resnet(x,training=False);x=layers.Dropout(.30,name='dropout')(x)
    out=layers.Dense(8,activation='softmax',dtype='float32',name='classifier')(x)
    return keras.Model(image,out,name=f'supervised_resnet50_seed_{seed}')
def load_image(path,y):
    blob=tf.io.read_file(path);x=tf.io.decode_image(blob,channels=3,expand_animations=False)
    x=tf.image.resize(x,[224,224],method='bilinear');x=tf.cast(x,tf.float32)/255.
    x.set_shape([224,224,3]);return x,y
BATCH=int(os.getenv('J11_BATCH','32'))
dataset=tf.data.Dataset.from_tensor_slices((cohort.filepath.to_numpy(dtype=str),cohort.label_id.to_numpy(dtype=np.int32))).map(load_image,num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)
all_prob=[];model_params={}
for seed in SEEDS:
    keras.backend.clear_session();gc.collect()
    model=build(seed);model.load_weights(str(ckpts[seed]));model_params[seed]=model.count_params()
    pr=np.asarray(model.predict(dataset,verbose=0),dtype=np.float64)
    req(pr.shape==(len(cohort),8) and np.isfinite(pr).all() and np.allclose(pr.sum(1),1,atol=1e-3),'Supervised predictions invalid')
    all_prob.append(pr);print('Loaded supervised seed',seed,':',pr.shape)
    del model;gc.collect()
p=np.stack(all_prob).mean(0);p=p/p.sum(1,keepdims=True)
np.savez_compressed(OUT/'predictions/j11_supervised_predictions.npz',
    image_id=cohort.image_id.to_numpy(dtype=str),hf_id=cohort.hf_id.to_numpy(dtype=str),
    y=cohort.label_id.to_numpy(dtype=np.int32),source=cohort.source.to_numpy(dtype=str),
    group_id=cohort.get('source_group',pd.Series(['']*len(cohort))).to_numpy(dtype=str),
    classes=np.array(CLASSES),proba=p.astype(np.float32),seeds=np.array(SEEDS))
write(OUT/'reports/trained_parameter_counts.json',{'Supervised_ResNet50_5seed':float(sum(model_params.values())/1e6),
    'Supervised_ResNet50_per_seed':float(next(iter(model_params.values()))/1e6)})
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score
rows=[]
for src in ['HAM10000','BCN20000']:
    mask=cohort.source.eq(src).to_numpy();y=cohort.label_id.to_numpy()[mask];pred=p[mask].argmax(1)
    rows.append({'source':src,'n':len(y),'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),
                'macro_f1_present':f1_score(y,pred,labels=np.unique(y),average='macro',zero_division=0)})
pd.DataFrame(rows).to_csv(OUT/'tables/supervised_ham_bcn.csv',index=False)
write(OUT/'reports/final_status.json',{'stage':'J11','status':'COMPLETED','images':len(cohort),
    'excluded_ambiguous_hf_members':len(ambiguous),'checkpoint_sha256':{str(s):sha(ckpts[s]) for s in SEEDS},
    'g02a_cohort_sha256':sha(cohort_path)})
print(pd.DataFrame(rows).to_string(index=False))


Loaded supervised seed 21 : (2239, 8)
Loaded supervised seed 35 : (2239, 8)
Loaded supervised seed 42 : (2239, 8)
Loaded supervised seed 52 : (2239, 8)
Loaded supervised seed 62 : (2239, 8)
  source    n  accuracy  balanced_accuracy  macro_f1_present
HAM10000  999  0.878879           0.775104          0.790874
BCN20000 1240  0.836290           0.808566          0.819753
